In [9]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import multiprocessing as mp
import torch.nn as nn
import os
import matplotlib.pyplot as plt
import itertools
import json

from tqdm import tqdm
from astropy.stats import sigma_clip
from scipy.optimize import minimize
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
from astropy.stats import sigma_clip
from sklearn.model_selection import train_test_split

In [10]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from pqdm.threads import pqdm
from scipy.optimize import minimize
from astropy.stats import sigma_clip
from scipy.signal import savgol_filter

import time
__t0 = time.perf_counter()
ROOT_PATH = "./data"
MODE = "train"

class Config:
    DATA_PATH = './data'
    DATASET = "train"
    OUTPUT_DIR = "./processed_data_features"

    SCALE = 0.95
    SIGMA = 0.0006
    
    CUT_INF = 39
    CUT_SUP = 321

    TEMPORAL_BINS = 30
    
    SENSOR_CONFIG = {
        "AIRS-CH0": {
            "raw_shape": [11250, 32, 356],
            "calibrated_shape": [1, 32, CUT_SUP - CUT_INF],
            "dark_shape": (32, 356),
            "dead_shape": (32, 356),
            "flat_shape": (32, 356),
            "linear_corr_shape": (6, 32, 356),
            "dt_pattern": (0.1, 4.5), 
            "binning": TEMPORAL_BINS
        },
        "FGS1": {
            "raw_shape": [135000, 32, 32],
            "calibrated_shape": [1, 32, 32],
            "dark_shape": (32, 32),
            "dead_shape": (32, 32),
            "flat_shape": (32, 32),
            "linear_corr_shape": (6, 32, 32),
            "dt_pattern": (0.1, 0.1),
            "binning": TEMPORAL_BINS * 12
        }
    }
    
    MODEL_PHASE_DETECTION_SLICE = slice(30, 140)
    MODEL_OPTIMIZATION_DELTA = 7
    MODEL_POLYNOMIAL_DEGREE = 3
    
    N_JOBS = 4

config = Config()



def _phase_detector_signal(signal, cfg):
    sl = cfg.MODEL_PHASE_DETECTION_SLICE
    min_idx = int(np.argmin(signal[sl])) + sl.start
    s1 = signal[:min_idx]; s2 = signal[min_idx:]
    if s1.size < 3 or s2.size < 3:
        return 0, len(signal) - 1
    g1 = np.gradient(s1); g1_max = np.max(g1) if np.size(g1) else 0.0
    g2 = np.gradient(s2); g2_max = np.max(g2) if np.size(g2) else 0.0
    if g1_max != 0: g1 /= g1_max
    if g2_max != 0: g2 /= g2_max
    phase1 = int(np.argmin(g1)); phase2 = int(np.argmax(g2)) + min_idx
    return phase1, phase2

def estimate_sigma_fgs(preprocessed_data, cfg):
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12
    for single in preprocessed_data:
        air_white = savgol_filter(single[:, 1:].mean(axis=1), 20, 2)
        p1, p2 = _phase_detector_signal(air_white, cfg)
        p1 = max(delta, p1)
        p2 = min(len(air_white) - delta - 1, p2)

        fgs = single[:, 0]
        oot = (fgs[: p1 - delta] if p1 - delta > 0 else np.empty(0, fgs.dtype))
        if p2 + delta < fgs.size:
            oot = np.concatenate([oot, fgs[p2 + delta :]])
        inn = fgs[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(fgs))
        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.8, 1.25)

    return k * cfg.SIGMA

def estimate_sigma_air(preprocessed_data, cfg):
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12

    for single in preprocessed_data:
        white = np.nanmean(single[:, 1:], axis=1)
        white_s = savgol_filter(white, 20, 2)

        p1, p2 = _phase_detector_signal(white_s, cfg)
        p1 = max(delta, p1)
        p2 = min(len(white) - delta - 1, p2)

        oot_left = white[: p1 - delta] if p1 - delta > 0 else np.empty(0, white.dtype)
        oot_right = white[p2 + delta :] if (p2 + delta) < white.size else np.empty(0, white.dtype)
        oot = np.concatenate([oot_left, oot_right]) if (oot_left.size + oot_right.size) else oot_left
        inn = white[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(white))

        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.90, 1.20)

    return k * cfg.SIGMA

In [11]:
import os
import numpy as np
import pandas as pd
from astropy.stats import sigma_clip
from tqdm.auto import tqdm

class SignalProcessor:
    def __init__(self, config):
        self.cfg = config
        self.adc_info = pd.read_csv(f"{self.cfg.DATA_PATH}/adc_info.csv")
        self.planet_ids = pd.read_csv(
            f'{self.cfg.DATA_PATH}/{self.cfg.DATASET}_star_info.csv',
            index_col='planet_id'
        ).index.astype(int)
        os.makedirs(self.cfg.OUTPUT_DIR, exist_ok=True)  # e.g. "./processed_data_features"

    def _apply_linear_corr(self, linear_corr, signal):
        # Horner scheme for polynomial correction
        coeffs = np.flip(linear_corr, axis=0)
        x = signal.astype(np.float64, copy=False)
        out = np.empty_like(x, dtype=np.float64)
        out[...] = coeffs[0]
        for k in range(1, coeffs.shape[0]):
            np.multiply(out, x, out=out)
            out += coeffs[k]
        return out.astype(signal.dtype, copy=False)

    def _calibrate_single_signal(self, planet_id, sensor, obs_id):
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]

        # Paths
        base = f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}"
        sig_pq = f"{base}/{sensor}_signal_{obs_id}.parquet"
        cal_dir = f"{base}/{sensor}_calibration_{obs_id}"
        dark_pq = f"{cal_dir}/dark.parquet"
        dead_pq = f"{cal_dir}/dead.parquet"
        flat_pq = f"{cal_dir}/flat.parquet"
        lin_pq  = f"{cal_dir}/linear_corr.parquet"

        if not (os.path.exists(sig_pq) and os.path.exists(dark_pq) and
                os.path.exists(dead_pq) and os.path.exists(flat_pq) and os.path.exists(lin_pq)):
            return None  # missing files -> skip observation

        # Loading and reshaping
        signal = pd.read_parquet(sig_pq).to_numpy().reshape(sensor_cfg["raw_shape"])
        dark   = pd.read_parquet(dark_pq).to_numpy().reshape(sensor_cfg["dark_shape"])
        dead   = pd.read_parquet(dead_pq).to_numpy().reshape(sensor_cfg["dead_shape"])
        flat   = pd.read_parquet(flat_pq).to_numpy().reshape(sensor_cfg["flat_shape"])
        linear_corr = pd.read_parquet(lin_pq).values.astype(np.float64).reshape(sensor_cfg["linear_corr_shape"])

        # ADC correction
        gain = self.adc_info[f"{sensor}_adc_gain"].iloc[0]
        offset = self.adc_info[f"{sensor}_adc_offset"].iloc[0]
        signal = signal / gain + offset

        # Hot/dead mask
        hot = sigma_clip(dark, sigma=5, maxiters=5).mask

        # AIRS: spectral crop
        if sensor == "AIRS-CH0":
            ci, cs = self.cfg.CUT_INF, self.cfg.CUT_SUP
            signal = signal[:, :, ci:cs]
            linear_corr = linear_corr[:, :, ci:cs]
            dark = dark[:, ci:cs]
            dead = dead[:, ci:cs]
            flat = flat[:, ci:cs]
            hot = hot[:, ci:cs]

        # FGS: ROI crop (y0:y1, x0:x1)
        if sensor == "FGS1":
            y0, y1, x0, x1 = 10, 22, 10, 22
            signal = signal[:, y0:y1, x0:x1]
            dark   = dark[y0:y1, x0:x1]
            dead   = dead[y0:y1, x0:x1]
            flat   = flat[y0:y1, x0:x1]
            linear_corr = linear_corr[:, y0:y1, x0:x1]
            hot    = hot[y0:y1, x0:x1]

        # Non negative
        np.maximum(signal, 0, out=signal)

        # Linearity correction
        if sensor == "FGS1":
            signal = self._apply_linear_corr(linear_corr, signal)
        elif sensor == "AIRS-CH0":
            sl = (slice(None), slice(10, 22), slice(None))  # T, Y, λ
            signal[sl] = self._apply_linear_corr(linear_corr[:, 10:22, :], signal[sl])
        else:
            signal = self._apply_linear_corr(linear_corr, signal)

        # Dark subtraction with dt-pattern
        base_dt, increment = sensor_cfg["dt_pattern"]
        even_scale = base_dt
        odd_scale  = base_dt + increment
        signal[::2] -= dark * even_scale
        signal[1::2] -= dark * odd_scale

        return signal

    def _preprocess_calibrated_signal(self, calibrated_signal, sensor, mode='mean'):
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]
        binning = sensor_cfg["binning"]

        # ROI + averaging over spatial dimensions
        if sensor == "AIRS-CH0":
            signal_roi = calibrated_signal[:, 10:22, :]              # (T, 12, λ)
        elif sensor == "FGS1":
            signal_roi = calibrated_signal[:, 10:22, 10:22]           # (T, 12, 12)
            signal_roi = signal_roi.reshape(signal_roi.shape[0], -1)  # (T, 144)
        mean_signal = np.nanmean(signal_roi, axis=1)                   # (T, λ) or (T, pixels)

        # CDS
        cds_signal = mean_signal[1::2] - mean_signal[0::2]             # (T/2, λ)

        # Temporal binning
        n_bins = cds_signal.shape[0] // binning
        if n_bins <= 0:
            return None
        if mode == 'median':
            binned = np.array([np.nanmedian(cds_signal[j*binning:(j+1)*binning], axis=0)
                               for j in range(n_bins)])
        else:
            binned = np.array([np.nanmean(cds_signal[j*binning:(j+1)*binning], axis=0)
                               for j in range(n_bins)])

        # AIRS: robust clipping per bin
        if sensor == "AIRS-CH0":
            q_lo = np.nanpercentile(binned, 5.0, axis=1, keepdims=True)
            q_hi = np.nanpercentile(binned, 95.0, axis=1, keepdims=True)
            np.clip(binned, q_lo, q_hi, out=binned)

        # FGS: to (n_bins, 1) collapse
        if sensor == "FGS1":
            binned = binned.reshape((binned.shape[0], 1))

        # AIRS: spectral weights (variance-inverse, robustly scaled)
        if sensor == "AIRS-CH0":
            var = np.nanvar(binned, axis=0, ddof=1)
            med = np.nanmedian(var)
            safe_var = np.where(~np.isfinite(var) | (var <= 0), med if (np.isfinite(med) and med > 0) else 1.0, var)
            w = 1.0 / safe_var
            lo, hi = np.nanpercentile(w, 5.0), np.nanpercentile(w, 95.0)
            if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
                w = np.clip(w, lo, hi)
            M = binned.shape[1]
            s = np.nansum(w)
            w = w * (M / s) if (np.isfinite(s) and s > 0) else np.ones_like(w)
            binned *= w[None, :]

        return binned

    def _process_single_observation(self, planet_id: int, obs_id: int, mode='mean'):
        # Calibrate and preprocess sensor signals
        fgs_cal = self._calibrate_single_signal(planet_id, "FGS1", obs_id)
        airs_cal = self._calibrate_single_signal(planet_id, "AIRS-CH0", obs_id)
        if fgs_cal is None or airs_cal is None:
            return None

        fgs_pre = self._preprocess_calibrated_signal(fgs_cal, "FGS1", mode=mode)   # (n_bins, 1)
        airs_pre = self._preprocess_calibrated_signal(airs_cal, "AIRS-CH0", mode=mode) # (n_bins, 282) (after Crop)
        if fgs_pre is None or airs_pre is None:
            return None

        # Align time axes
        n = min(fgs_pre.shape[0], airs_pre.shape[0])
        fgs_pre = fgs_pre[:n]
        airs_pre = airs_pre[:n]

        # Combine: (n_bins, 1 + 282) = (n_bins, 283)
        combined = np.concatenate([fgs_pre, airs_pre], axis=1).astype(np.float32)

        # Save
        out_path = os.path.join(self.cfg.OUTPUT_DIR, f"planet_{planet_id}_signal_{obs_id}.npy")
        np.save(out_path, combined)
        return out_path

    def process_all_data(self, mode='mean'):
        tasks = []
        for pid in self.planet_ids:
            for obs_id in (0, 1):
                base = f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{pid}"
                if (os.path.exists(f"{base}/FGS1_signal_{obs_id}.parquet") and
                    os.path.exists(f"{base}/AIRS-CH0_signal_{obs_id}.parquet")):
                    tasks.append((int(pid), int(obs_id), mode))

        results = []
        for pid, obs_id, m in tqdm(tasks, desc="Processing observations (seq)"):
            out = self._process_single_observation(pid, obs_id, mode=m)
            if out is not None:
                results.append(out)
        return results


In [12]:
# config = Config()
# sp = SignalProcessor(config)
# saved_paths = sp.process_all_data(mode='mean')
# print(f"Saved {len(saved_paths)} files:")
# for p in saved_paths[:5]:
#     print("  ", p)

In [13]:
import os
import glob
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict, Optional, Iterable


class ProcessedObsLoader:

    def __init__(self, data_dir: str = "./processed_data_features", mmap: bool = False):
        self.data_dir = data_dir
        self.mmap = mmap

        pattern = os.path.join(data_dir, "planet_*_signal_*.npy")
        files = sorted(glob.glob(pattern))

        # (planet_id, obs_id, path)
        self.index: List[Tuple[int, int, str]] = []
        for p in files:
            name = os.path.basename(p)
            try:
                # Expect "planet_{pid}_signal_{obs}.npy"
                pid = int(name.split("_")[1])
                obs = int(os.path.splitext(name)[0].split("_")[-1])
            except Exception:
                continue
            self.index.append((pid, obs, p))

        # planet -> sorted obs list
        self.by_planet: Dict[int, List[int]] = {}
        for pid, obs, _ in self.index:
            self.by_planet.setdefault(pid, []).append(obs)
        for pid in self.by_planet:
            self.by_planet[pid] = sorted(set(self.by_planet[pid]))

    def __len__(self) -> int:
        return len(self.index)

    def list_planets(self) -> List[int]:
        return sorted(self.by_planet.keys())

    def list_observations(self, planet_id: int) -> List[int]:
        return self.by_planet.get(int(planet_id), [])

    def _np_load(self, path: str) -> np.ndarray:
        if self.mmap:
            return np.load(path, mmap_mode="r")
        return np.load(path)

    def load_single_observation(self, planet_id: int, obs_id: int) -> np.ndarray:
        matches = [p for (pid, obs, p) in self.index if pid == int(planet_id) and obs == int(obs_id)]
        if not matches:
            raise FileNotFoundError(f"No file for planet={planet_id}, obs={obs_id} in {self.data_dir}")
        return self._np_load(matches[0])

    def get_obs_iterator(self) -> Iterable[Tuple[int, int, np.ndarray]]:
        """Yield (planet_id, obs_id, array) lazily."""
        for pid, obs, p in self.index:
            yield pid, obs, self._np_load(p)

    def get_planet_iterator(self, planet_id: int) -> Iterable[Tuple[int, int, np.ndarray]]:
        """Yield all observations for one planet."""
        for pid, obs, p in self.index:
            if pid == int(planet_id):
                yield pid, obs, self._np_load(p)

    # Simplified: no filter / no stack; split by obs_id for train/val
    def load_all_observations(self):
        """
        Returns:
          train_transit_features: list of np.ndarray (T, 283) for obs_id == 0
          val_transit_features:   list of np.ndarray (T, 283) for obs_id == 1
          val_indices:            np.ndarray of planet_id matching val list order
        """
        train_data: List[np.ndarray] = []
        train_idx:  List[int]        = []
        val_data:   List[np.ndarray] = []
        val_idx:    List[int]        = []

        for pid, obs, p in tqdm(self.index, desc="Loading train/val by obs_id"):
            if obs == 0:
                train_data.append(self._np_load(p))
                train_idx.append(pid)
            elif obs == 1:
                val_data.append(self._np_load(p))
                val_idx.append(pid)
            else:
                # ignore other obs_ids
                continue

        return np.asarray(train_data), np.asarray(train_idx, dtype=np.int64), np.asarray(val_data), np.asarray(val_idx, dtype=np.int64)

In [14]:
loader = ProcessedObsLoader("./processed_data_features", mmap=False)

train_planets, train_indices, val_planets, val_indices = loader.load_all_observations()
train_planets.shape, train_indices.shape, val_planets.shape, val_indices.shape

Loading train/val by obs_id: 100%|██████████| 1210/1210 [00:00<00:00, 7811.46it/s]


((1100, 187, 283), (1100,), (110, 187, 283), (110,))

In [15]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.signal import savgol_filter
from scipy.optimize import minimize


class TransitModelWithFeaturesPerFreq:
    def __init__(self, config):
        self.cfg = config

    def _phase_detector(self, signal):
        search_slice = self.cfg.MODEL_PHASE_DETECTION_SLICE
        min_index = np.argmin(signal[search_slice]) + search_slice.start

        signal1 = signal[:min_index]
        signal2 = signal[min_index:]

        grad1 = np.gradient(signal1)
        grad1 /= grad1.max()

        grad2 = np.gradient(signal2)
        grad2 /= grad2.max()

        phase1 = np.argmin(grad1)
        phase2 = np.argmax(grad2) + min_index
        return phase1, phase2

    def _objective_function(self, s, signal, phase1, phase2):
        delta = self.cfg.MODEL_OPTIMIZATION_DELTA
        power = self.cfg.MODEL_POLYNOMIAL_DEGREE

        if phase1 - delta <= 0 or phase2 + delta >= len(signal) or (phase2 - delta) - (phase1 + delta) < 5:
            delta = 2

        y = np.concatenate([
            signal[: phase1 - delta],
            signal[phase1 + delta : phase2 - delta] * (1 + s),
            signal[phase2 + delta :]
        ])
        x = np.arange(len(y))

        coeffs = np.polyfit(x, y, deg=power)
        poly = np.poly1d(coeffs)
        error = np.abs(poly(x) - y).mean()
        return error

    # ---- per-frequency feature extraction ----
    def _predict_single_channel(self, signal_1d):
        # 1) smooth
        signal_1d_smooth = savgol_filter(signal_1d, 20, 2)
        T = len(signal_1d_smooth)

        # 2) phases
        phase1, phase2 = self._phase_detector(signal_1d_smooth)

        # 3) optimize s (depth)
        bounded_phase1 = max(self.cfg.MODEL_OPTIMIZATION_DELTA, phase1)
        bounded_phase2 = min(T - self.cfg.MODEL_OPTIMIZATION_DELTA - 1, phase2)

        result = minimize(
            fun=self._objective_function,
            x0=[0.0001],
            args=(signal_1d_smooth, bounded_phase1, bounded_phase2),
            method="Nelder-Mead"
        )
        s_hat = result.x[0] if result.success else 0.0
        depth = s_hat * self.cfg.SCALE

        # 4) side features (same logic)
        delta = self.cfg.MODEL_OPTIMIZATION_DELTA
        if bounded_phase1 - delta <= 0 or bounded_phase2 + delta >= T or (bounded_phase2 - delta) - (bounded_phase1 + delta) < 5:
            delta = 2

        i0, i1 = bounded_phase1 + delta, bounded_phase1 - delta
        e0, e1 = bounded_phase2 - delta, bounded_phase2 + delta

        oot_mask = np.zeros(T, bool); oot_mask[:i1] = True; oot_mask[e1:] = True
        it_mask  = np.zeros(T, bool); it_mask[i0:e0] = True

        baseline  = signal_1d_smooth[oot_mask].mean() if oot_mask.any() else 1.0
        rms_out   = signal_1d_smooth[oot_mask].std() + 1e-12 if oot_mask.any() else 1e-12
        rms_in    = signal_1d_smooth[it_mask].std() if it_mask.any() else 0.0
        rms_ratio = rms_in / rms_out

        d1_smooth = np.gradient(signal_1d_smooth)
        d2_smooth = np.gradient(d1_smooth)
        w = max(3, delta * 2 + 1)
        curv_ing = np.nanmean(np.abs(d2_smooth[max(0, bounded_phase1 - w):min(T, bounded_phase1 + w)]))
        curv_egr = np.nanmean(np.abs(d2_smooth[max(0, bounded_phase2 - w):min(T, bounded_phase2 + w)]))

        eqw_slice = slice(max(0, i0), min(T, e0))
        eqw = np.sum((baseline - signal_1d_smooth[eqw_slice]) / baseline) if eqw_slice.start < eqw_slice.stop else 0.0

        return {
            "depth": float(depth),
            "baseline": float(baseline),
            "rms_in": float(rms_in),
            "rms_out": float(rms_out),
            "rms_ratio": float(rms_ratio),
            "curv_ing": float(curv_ing),
            "curv_egr": float(curv_egr),
            "eqw": float(eqw),
            "T14_frames": float(phase2 - phase1),
            "phase1": float(phase1),
            "phase2": float(phase2),
        }

    def predict_per_frequency(self, single_preprocessed_signal):
        """
        Input: (T, 1+F) or (T, F) where col0 may be time/ignored.
        Output: list of dicts length F.
        """
        # decide where frequency channels start
        if single_preprocessed_signal.ndim != 2:
            raise ValueError("Expected 2D array (T, C).")

        T, C = single_preprocessed_signal.shape
        # If there's a known leading column to ignore (like time), skip it
        start_col = 1 if C > 1 else 0
        F = C - start_col
        feats_per_freq = []
        for f in range(F):
            sig = single_preprocessed_signal[:, start_col + f]
            feats_per_freq.append(self._predict_single_channel(sig))
        return feats_per_freq  # length F, each a dict

    def predict_all(self, preprocessed_signals):
        """
        Input: (N, T, C) -> returns (features_array, feature_names)
        features_array shape: (N, F, K) with fixed feature order.
        """
        feature_keys = [
            "depth", "baseline", "rms_in", "rms_out", "rms_ratio",
            "curv_ing", "curv_egr", "eqw", "T14_frames", "phase1", "phase2"
        ]

        N, T, C = preprocessed_signals.shape
        start_col = 1 if C > 1 else 0
        F = C - start_col
        K = len(feature_keys)

        out = np.zeros((N, F, K), dtype=np.float32)

        for i in tqdm(range(N), desc="Extracting Features per-frequency"):
            feats_list = self.predict_per_frequency(preprocessed_signals[i])
            # map dicts to ordered vector
            for f in range(F):
                vec = [feats_list[f][k] for k in feature_keys]
                out[i, f, :] = np.asarray(vec, dtype=np.float32)

        return out, feature_keys


# transit_model = TransitModelWithFeaturesPerFreq(config)
# train_transit_features, _ = transit_model.predict_all(train_planets)
# val_transit_features, _ = transit_model.predict_all(val_planets)
# np.save('./processed_data_features/train_transit_features.npy', train_transit_features)
# np.save('./processed_data_features/val_transit_features.npy', val_transit_features)

train_transit_features = np.load('./processed_data_features/train_transit_features.npy', allow_pickle=False)
val_transit_features = np.load('./processed_data_features/val_transit_features.npy', allow_pickle=False)
train_transit_features.shape, val_transit_features.shape

((1100, 282, 11), (110, 282, 11))

In [16]:
train_star_info = pd.read_csv(ROOT_PATH + f"/{MODE}_star_info.csv", index_col='planet_id').loc[train_indices]
val_star_info = train_star_info.loc[val_indices]
# star_info_columns = ['Rs', 'Ms', 'Ts', 'Mp', 'e', 'P', 'sma', 'i']
star_info_columns = ['Rs', 'i']

train_transit_features = np.concatenate((train_transit_features.reshape(train_transit_features.shape[0], -1), train_star_info[star_info_columns].values), axis=1)
val_transit_features = np.concatenate((val_transit_features.reshape(val_transit_features.shape[0], -1), val_star_info[star_info_columns].values), axis=1)
train_transit_features.shape, val_transit_features.shape

((1100, 3104), (110, 3104))

In [17]:
train_labels = pd.read_csv('./data/train.csv', index_col='planet_id').loc[train_indices]
val_labels = train_labels.loc[val_indices]
train_labels, val_labels = train_labels.values, val_labels.values
train_labels.shape, val_labels.shape

((1100, 283), (110, 283))

# Feature Selection

In [18]:
# feature_selection_pipeline.py
import numpy as np
from dataclasses import dataclass
from typing import Optional, Dict, Any

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, SelectFromModel, RFECV
from sklearn.linear_model import MultiTaskLassoCV, Ridge
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold, GroupKFold
import joblib


@dataclass
class FeatureSelectionMeta:
    stage1_pipeline: Pipeline
    rfecv: RFECV
    perm_keep_mask: np.ndarray
    final_mask_in_original_space: np.ndarray
    scaler_before_stage1: Optional[StandardScaler] = None
    info: Optional[Dict[str, Any]] = None


class FeatureSelector:
    """
    Two-stage, leakage-free feature selection:
      - Stage 1: VarianceThreshold + SelectFromModel(MultiTaskLassoCV)
      - Stage 2: RFECV(Ridge) to pick subset size
      - Stage 3: Permutation importance on held-out validation to prune train-only features
    """
    def __init__(self, cv_splits=5, random_state=42, step=0.1, scoring="neg_mean_squared_error",
                 use_groups: bool = False):
        self.cv_splits = cv_splits
        self.random_state = random_state
        self.step = step
        self.scoring = scoring
        self.use_groups = use_groups

        self.meta: Optional[FeatureSelectionMeta] = None

    def _build_stage1(self):
        base = MultiTaskLassoCV(cv=self.cv_splits, n_jobs=-1, random_state=self.random_state, verbose=True)
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("var", VarianceThreshold(0.0)),
            ("sfm", SelectFromModel(base, max_features=None, prefit=False))
        ])
        return pipe

    def fit_transform(self, X_train: np.ndarray, Y_train: np.ndarray,
                      X_val: np.ndarray, Y_val: np.ndarray,
                      groups_train: Optional[np.ndarray] = None):
        # Stage 1
        stage1 = self._build_stage1()
        if self.use_groups and groups_train is not None:
            # MultiTaskLassoCV inside Pipeline handles cv internally; groups not supported directly there
            # We rely on shuffle KFold at stage2; keep group split for RFECV below
            pass
        X1_tr = stage1.fit_transform(X_train, Y_train)
        X1_va = stage1.transform(X_val)

        # Stage 2: RFECV
        if self.use_groups and groups_train is not None:
            cv = GroupKFold(n_splits=self.cv_splits)
            cv_arg = cv.split(X1_tr, Y_train, groups_train)
        else:
            cv = KFold(n_splits=self.cv_splits, shuffle=True, random_state=self.random_state)
            cv_arg = cv

        rfecv = RFECV(
            estimator=Ridge(alpha=1.0, random_state=self.random_state),
            step=self.step,
            cv=cv_arg,
            scoring=self.scoring,
            n_jobs=-1
        )
        rfecv.fit(X1_tr, Y_train)
        X2_tr = rfecv.transform(X1_tr)
        X2_va = rfecv.transform(X1_va)

        # Stage 3: Permutation importance on held-out validation
        est = Ridge(alpha=1.0, random_state=self.random_state).fit(X2_tr, Y_train)
        perm = permutation_importance(est, X2_va, Y_val, n_repeats=20,
                                      random_state=self.random_state, scoring=self.scoring)
        keep_perm = perm.importances_mean > np.maximum(0.0, 2 * perm.importances_std)
        if keep_perm.sum() == 0:
            keep_perm = perm.importances_mean > 0.0
        Xf_tr = X2_tr[:, keep_perm]
        Xf_va = X2_va[:, keep_perm]

        # Compose mask back to original D
        var_mask = stage1.named_steps["var"].get_support()
        sfm_mask = stage1.named_steps["sfm"].get_support()
        mask_stage1 = np.zeros_like(var_mask, dtype=bool); mask_stage1[var_mask] = sfm_mask
        mask_stage2 = np.zeros_like(mask_stage1, dtype=bool); mask_stage2[mask_stage1] = rfecv.support_
        final_mask = np.zeros_like(mask_stage2, dtype=bool); final_mask[mask_stage2] = keep_perm

        self.meta = FeatureSelectionMeta(
            stage1_pipeline=stage1,
            rfecv=rfecv,
            perm_keep_mask=keep_perm,
            final_mask_in_original_space=final_mask,
            info={
                "X1_train_dim": X1_tr.shape[1],
                "X2_train_dim": X2_tr.shape[1],
                "X_final_dim": Xf_tr.shape[1],
            }
        )
        return Xf_tr, Xf_va

    def transform(self, X: np.ndarray) -> np.ndarray:
        assert self.meta is not None, "fit_transform first"
        X1 = self.meta.stage1_pipeline.transform(X)
        X2 = self.meta.rfecv.transform(X1)
        Xf = X2[:, self.meta.perm_keep_mask]
        return Xf

    def save(self, path: str):
        assert self.meta is not None, "nothing to save"
        joblib.dump(self.meta, path)

    def load(self, path: str):
        self.meta = joblib.load(path)
        return self


# Model for mu-values

In [19]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

class ArielDataset(Dataset):
    def __init__(self, features, targets=None):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.Y = torch.tensor(targets, dtype=torch.float32) if targets is not None else None
        self.is_labeled = targets is not None
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.is_labeled:
            return self.X[idx], self.Y[idx]
        else:
            return self.X[idx]


class MLPModel(nn.Module):
    def __init__(self, input_dim=3, output_dim=283):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            # nn.ReLU(),
            # nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)
    

def normalized_gll_score(preds, targets, sigma_fgs_vec, sigma_air_vec,
                         naive_mean=0.01468902, naive_sigma=0.01066135,
                         fgs_weight=57.846, fgs_sigma_true=1e-6, airs_sigma_true=1e-5):
    """
    Calculate normalized GLL score using adaptive sigmas (NOT for backprop).
    This gives us the real competition metric to monitor training progress.
    """
    eps = 1e-15
    B, n_wavelengths = preds.shape
    
    # Create sigma_pred tensor using estimated uncertainties per planet
    sigma_pred = torch.empty_like(preds)
    sigma_pred[:, 0] = torch.tensor(sigma_fgs_vec, dtype=preds.dtype, device=preds.device)
    sigma_pred[:, 1:] = torch.tensor(sigma_air_vec, dtype=preds.dtype, device=preds.device).unsqueeze(1).expand(B, n_wavelengths-1)
    sigma_pred = torch.clamp(sigma_pred, min=eps)
    
    # True sigma values for normalization
    sigma_true = torch.cat([
        preds.new_full((1,), fgs_sigma_true),
        preds.new_full((n_wavelengths-1,), airs_sigma_true)
    ]).unsqueeze(0).expand(B, -1)
    
    def logpdf(x, mu, sigma):
        return -0.5 * ((x - mu) / sigma).pow(2) - torch.log(sigma) - 0.5 * np.log(2*np.pi)
    
    # Calculate log likelihoods exactly like official implementation
    GLL_pred = logpdf(targets, preds, sigma_pred)
    GLL_true = logpdf(targets, targets, sigma_true)
    
    # Naive baseline using provided train set statistics
    naive_mean_tensor = torch.tensor(naive_mean, dtype=preds.dtype, device=preds.device)
    naive_sigma_tensor = torch.tensor(naive_sigma, dtype=preds.dtype, device=preds.device)
    GLL_mean = logpdf(targets, naive_mean_tensor, naive_sigma_tensor)
    
    # Normalized individual scores
    ind_scores = (GLL_pred - GLL_mean) / (GLL_true - GLL_mean)
    
    # Weights: FGS1 vs AIRS
    weights = torch.cat([
        preds.new_full((1,), fgs_weight),
        torch.ones(n_wavelengths-1, device=preds.device)
    ]).unsqueeze(0).expand_as(ind_scores)
    
    # Weighted average score
    weighted_score = (ind_scores * weights).sum()/weights.sum()
    
    return weighted_score.item()


def train_model(train_X, train_Y, train_sigma_vec,
                val_X=None, val_Y=None, val_sigma_vec=None,
                groups_train=None, use_groups=False, seed=42,
                num_epochs=100, lr_patience=10, lr_cooldown=5, lr=1e-3, batch_size=64,
                checkpoint_path='./checkpoints_gll', feature_selector_path='./processed_data_features/feature_selector.pkl'):

    os.makedirs(checkpoint_path, exist_ok=True)

    # fs = FeatureSelector(cv_splits=5, random_state=seed, use_groups=use_groups)
    # train_X_selected, val_X_selected = fs.fit_transform(train_X[:200], train_Y[:200], val_X[:50], val_Y[:50], groups_train=groups_train)
    # fs.save(feature_selector_path)

    # feature_selector = joblib.load(feature_selector_path)
    # var_mask = feature_selector.stage1_pipeline.named_steps["var"].get_support()
    # sfm_mask = feature_selector.stage1_pipeline.named_steps["sfm"].get_support()
    # stage1_mask = np.zeros_like(var_mask, dtype=bool)
    # stage1_mask[var_mask] = sfm_mask
    # stage1_indices = np.where(stage1_mask)[0]
    # stage3_indices = np.where(feature_selector.final_mask_in_original_space)[0]
    
    # train_X_selected = train_X[:, stage3_indices]
    # if val_X is not None:
    #     val_X_selected = val_X[:, stage3_indices]

    train_X_selected = train_X
    if val_X is not None:
        val_X_selected = val_X

    scaler = StandardScaler()
    train_X_scaled = scaler.fit_transform(train_X_selected)
    if val_X is not None:
        val_X_scaled = scaler.transform(val_X_selected)
    scaler_path = os.path.join(checkpoint_path, 'scaler.pkl')
    joblib.dump(scaler, scaler_path)


    train_ds = ArielDataset(torch.from_numpy(train_X_scaled).float(),
                             torch.from_numpy(np.asarray(train_Y)).float())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    has_val = (val_X is not None) and (val_Y is not None) and (val_sigma_vec is not None)
    if has_val:
        val_X_tensor = torch.from_numpy(val_X_scaled).float()
        val_Y_tensor = torch.from_numpy(np.asarray(val_Y)).float()
        val_sigma_fgs = np.asarray(val_sigma_vec['fgs_sigma'])
        val_sigma_air = np.asarray(val_sigma_vec['airs_sigma'])

    input_dim = train_X_scaled.shape[1]
    output_dim = train_Y.shape[1]
    model = MLPModel(input_dim=input_dim, output_dim=output_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=lr_patience, factor=0.5, threshold=1e-9,
        cooldown=lr_cooldown
    )

    mse_fn = nn.MSELoss()
    best_metric = float('inf')
    best_gll = -float('inf')
    epochs_no_improve = 0

    print(f"Training started: {len(train_ds)} samples, {input_dim}→{output_dim}")

    for epoch in range(num_epochs):
        model.train()
        total_mse_loss = 0.0
        total_gll_score = 0.0
        num_samples = 0

        for batch_X, batch_Y in train_loader:
            optimizer.zero_grad()
            pred = model(batch_X)
            mse_loss = mse_fn(pred, batch_Y)
            mse_loss.backward()
            optimizer.step()

            with torch.no_grad():
                Nsig = len(train_sigma_vec['fgs_sigma'])
                take = min(batch_X.size(0), Nsig)
                idx = torch.randperm(Nsig)[:take]
                sig_fgs = np.asarray(train_sigma_vec['fgs_sigma'])[idx.numpy()]
                sig_air = np.asarray(train_sigma_vec['airs_sigma'])[idx.numpy()]
                gll = normalized_gll_score(pred[:take], batch_Y[:take], sig_fgs, sig_air)

            bs = batch_X.size(0)
            total_mse_loss += mse_loss.item() * bs
            total_gll_score += float(gll) * bs
            num_samples += bs

        avg_train_mse = total_mse_loss / max(1, num_samples)
        avg_train_gll = total_gll_score / max(1, num_samples)

        if has_val:
            model.eval()
            with torch.no_grad():
                val_pred = model(val_X_tensor)
                val_mse = mse_fn(val_pred, val_Y_tensor).item()
                val_gll = normalized_gll_score(val_pred, val_Y_tensor, val_sigma_fgs, val_sigma_air)
        else:
            val_mse, val_gll = None, None

        metric_for_sched = val_mse if has_val else avg_train_mse
        scheduler.step(metric_for_sched)

        is_better = (val_mse < best_metric) if has_val else (avg_train_mse < best_metric)
        if is_better:
            best_metric = val_mse if has_val else avg_train_mse
            best_gll = val_gll if has_val else avg_train_gll
            epochs_no_improve = 0

            checkpoint = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_path': scaler_path,
                'mse_loss': best_metric,
                'gll_score': best_gll,
                'epoch': epoch,
                'input_dim': input_dim,
                'output_dim': output_dim,
                'use_val': has_val,
            }
            model_path = os.path.join(checkpoint_path, 'gll_model_best.pth')
            torch.save(checkpoint, model_path)
        else:
            epochs_no_improve += 1


        if epochs_no_improve >= max(1, num_epochs // 3):
            print(f"Early stopping at epoch {epoch+1}")
            break

        if epoch % 10 == 0 or epoch == num_epochs - 1:
            lr_current = optimizer.param_groups[0]['lr']
            if has_val:
                print(f"Epoch {epoch+1:3d}/{num_epochs} | "
                      f"Train MSE: {avg_train_mse} | Train GLL: {avg_train_gll:.4f} | "
                      f"Val MSE: {val_mse} | Val GLL: {val_gll:.4f} | "
                      f"LR: {lr_current:.6f} | Best: {best_metric}")
            else:
                print(f"Epoch {epoch+1:3d}/{num_epochs} | "
                      f"Train MSE: {avg_train_mse} | Train GLL: {avg_train_gll:.4f} | "
                      f"LR: {lr_current:.6f} | Best: {best_metric}")

    final_path = os.path.join(checkpoint_path, f"gll_model_{best_metric}.pth")
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_path': scaler_path,
        'mse_loss': best_metric,
        'gll_score': best_gll,
        'epoch': epoch,
        'input_dim': input_dim,
        'output_dim': output_dim,
        'use_val': has_val,
    }, final_path)

    print(f"Training completed — best metric: {best_metric}, GLL: {best_gll:.6f}")
    return model, scaler



def load_model(checkpoint_path='./checkpoints_gll/gll_model_best.pth'):
    """Load trained model and scaler"""

    checkpoint = torch.load(checkpoint_path, map_location='cpu')

    # Recreate model
    model = MLPModel(
        input_dim=checkpoint['input_dim'],
        output_dim=checkpoint['output_dim']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Load scaler
    scaler = joblib.load(checkpoint['scaler_path'])
    
    print(f"Model loaded: MSE {checkpoint['mse_loss']}, GLL {checkpoint['gll_score']:.4f}")
    
    return model, scaler


def predict_with_model(model, scaler, X_test):
    """Make predictions on test data"""
    X_scaled = scaler.transform(X_test)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    
    model.eval()
    with torch.no_grad():
        predictions = model(X_tensor).numpy()
    
    return predictions

In [58]:
print("Training MLP to learn mu-values...")

train_sigma_estimates = pd.read_csv("./data/sigma_estimates.csv", index_col='planet_id').loc[train_indices]
val_sigma_features = train_sigma_estimates.loc[val_indices]

mu_model, scaler = train_model(
    train_X=train_transit_features.astype(np.float32),
    train_Y=train_labels.astype(np.float32),
    train_sigma_vec=train_sigma_estimates,
    val_X=val_transit_features.astype(np.float32),
    val_Y=val_labels.astype(np.float32),
    val_sigma_vec=val_sigma_features,
    num_epochs=10000,
    lr_patience=200,
    lr_cooldown=0,
    lr=1e-3,
    batch_size=1024,
    checkpoint_path='./checkpoints_gll_mean',
    feature_selector_path='./processed_data_features/feature_selector.pkl'
)


Training MLP to learn mu-values...
Training started: 1100 samples, 3104→283
Epoch   1/10000 | Train MSE: 0.07860895406116139 | Train GLL: -12579.0004 | Val MSE: 0.12931454181671143 | Val GLL: -17196.8027 | LR: 0.001000 | Best: 0.12931454181671143


C:\Users\fazul\AppData\Local\Temp\ipykernel_6460\3406241572.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.X = torch.tensor(features, dtype=torch.float32)
C:\Users\fazul\AppData\Local\Temp\ipykernel_6460\3406241572.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.Y = torch.tensor(targets, dtype=torch.float32) if targets is not None else None


Epoch  11/10000 | Train MSE: 0.020729016641324215 | Train GLL: -3789.3544 | Val MSE: 0.01600443571805954 | Val GLL: -2941.8479 | LR: 0.001000 | Best: 0.01600443571805954
Epoch  21/10000 | Train MSE: 0.007572764481671832 | Train GLL: -1340.3892 | Val MSE: 0.008116503246128559 | Val GLL: -1373.0798 | LR: 0.001000 | Best: 0.008116503246128559
Epoch  31/10000 | Train MSE: 0.0039023761857639658 | Train GLL: -682.8904 | Val MSE: 0.0061892010271549225 | Val GLL: -1035.7271 | LR: 0.001000 | Best: 0.0061892010271549225
Epoch  41/10000 | Train MSE: 0.002501649893820286 | Train GLL: -434.9975 | Val MSE: 0.005350607912987471 | Val GLL: -863.3343 | LR: 0.001000 | Best: 0.005350607912987471
Epoch  51/10000 | Train MSE: 0.001829700156284327 | Train GLL: -315.6253 | Val MSE: 0.004821844398975372 | Val GLL: -779.4952 | LR: 0.001000 | Best: 0.004821844398975372
Epoch  61/10000 | Train MSE: 0.0013735505129972643 | Train GLL: -236.3383 | Val MSE: 0.004638511221855879 | Val GLL: -728.7458 | LR: 0.001000 | 

KeyboardInterrupt: 

# Model for sigma-values

In [20]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ====== Stable normalized GLL loss (no .item, backprop-safe) ======
def normalized_gll_loss_from_sigma(preds, targets, sigma_pred,
                                   naive_mean=0.01468902, naive_sigma=0.01066135,
                                   fgs_weight=57.846, fgs_sigma_true=1e-6, airs_sigma_true=1e-5,
                                   eps=1e-12):
    """
    Returns a scalar loss = -normalized_GLL (mean-weighted over batch and wavelengths),
    suitable for backprop. All tensors are torch tensors on the same device.
    """
    preds = preds.float(); targets = targets.float()
    sigma_pred = torch.clamp(sigma_pred.float(), min=eps)

    B, F = preds.shape
    two_pi = preds.new_tensor(2 * np.pi)

    def logpdf(x, mu, sigma):
        return -0.5 * ((x - mu) / sigma) ** 2 - torch.log(sigma) - 0.5 * torch.log(two_pi)

    GLL_pred = logpdf(targets, preds, sigma_pred)

    sigma_true = torch.cat([
        preds.new_full((1,), fgs_sigma_true),
        preds.new_full((F-1,), airs_sigma_true)
    ], dim=0).view(1, F).expand(B, -1)
    GLL_true = logpdf(targets, targets, sigma_true)
    GLL_mean = logpdf(targets,
                      preds.new_full((B, F), naive_mean),
                      preds.new_full((B, F), naive_sigma))

    denom = torch.clamp(GLL_true - GLL_mean, min=eps)
    ind_scores = (GLL_pred - GLL_mean) / denom  # (B,F)

    w = torch.cat([
        preds.new_full((1,), fgs_weight),
        torch.ones(F-1, device=preds.device)
    ], dim=0).view(1, F).expand_as(ind_scores)

    # Average like official metric
    score = (ind_scores * w).sum() / w.sum()  # scalar
    loss = -score  # maximize score -> minimize negative score
    return loss, score  # return both for logging


# ====== Frozen trunk + sigma head with instrument floor ======
class FrozenTrunkSigmaHead(nn.Module):
    def __init__(self, mu_model: nn.Module, out_dim=283, head_hidden=256,
                 fgs_floor=1e-6, airs_floor=1e-5):
        super().__init__()
        layers = list(mu_model.net.children())
        assert isinstance(layers[-1], nn.Linear), "Last layer must be Linear in the μ model"

        self.trunk = nn.Sequential(*layers[:-1])
        for p in self.trunk.parameters():
            p.requires_grad = False

        trunk_out = layers[-1].in_features
        self.head = nn.Sequential(
            nn.Linear(trunk_out, head_hidden),
            # nn.GELU(),
            # nn.Linear(head_hidden, head_hidden),
            # nn.GELU(),
            # nn.Dropout(0.1),
            # nn.Linear(head_hidden, head_hidden),
            # nn.GELU(),
            # nn.Dropout(0.15),
            # nn.Linear(head_hidden, head_hidden),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(head_hidden, out_dim)
        )

        floor = torch.cat([
            torch.tensor([fgs_floor], dtype=torch.float32),
            torch.full((out_dim-1,), airs_floor, dtype=torch.float32)
        ], dim=0)
        self.register_buffer("sigma_floor", floor)
        self.softplus = nn.Softplus()

    def forward(self, x):
        raw = self.head(self.trunk(x))
        sp  = self.softplus(raw)
        sigma = torch.sqrt(sp**2 + self.sigma_floor.view(1, -1)**2)
        return sigma


# ====== Dataset (no residual labels needed when training on GLL) ======
class SigmaGLLDataset(Dataset):
    """
    Provides X and Y for GLL-based sigma training.
    μ̂ is computed on the fly (frozen μ-model) in the training loop for correct alignment.
    """
    def __init__(self, X_scaled: np.ndarray, Y: np.ndarray):
        self.X = torch.from_numpy(X_scaled.astype(np.float32))
        self.Y = torch.from_numpy(Y.astype(np.float32))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.Y[i]


# ====== Trainer that optimizes normalized GLL directly ======
def train_sigma_head(mu_model, scaler,
                            train_X, train_Y,
                            val_X=None, val_Y=None,
                            num_epochs=200, batch_size=512, lr=1e-3, weight_decay=1e-5,
                            fgs_weight=57.846,
                            naive_mean=0.01468902, naive_sigma=0.01066135,
                            fgs_sigma_true=1e-6, airs_sigma_true=1e-5,
                            ckpt_dir="./checkpoints_gll_mean",
                            lr_patience=200):
    os.makedirs(ckpt_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Scale features
    train_X_scaled = scaler.transform(train_X)
    val_X_scaled = scaler.transform(val_X) if val_X is not None else None

    # Datasets
    train_ds = SigmaGLLDataset(train_X_scaled, train_Y)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    has_val = (val_X is not None) and (val_Y is not None)
    if has_val:
        val_ds = SigmaGLLDataset(val_X_scaled, val_Y)
        val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    # Models
    mu_model = mu_model.to(device).eval()  # frozen μ
    sigma_model = FrozenTrunkSigmaHead(mu_model).to(device)

    opt = optim.AdamW(filter(lambda p: p.requires_grad, sigma_model.parameters()), lr=lr, weight_decay=weight_decay)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=lr_patience, threshold=1e-7)

    best_val = float("inf")
    best_path = os.path.join(ckpt_dir, "sigma_head_best.pt")

    for ep in range(1, num_epochs+1):
        sigma_model.train()
        tot_loss, tot_score, n = 0.0, 0.0, 0

        for xb, yb in train_dl:
            xb = xb.to(device); yb = yb.to(device)
            opt.zero_grad()

            # Forward μ̂ and σ̂
            with torch.no_grad():
                mu_b = mu_model(xb)  # (B, F)
            sigma_b = sigma_model(xb)

            # GLL-based loss
            loss_b, score_b = normalized_gll_loss_from_sigma(
                mu_b, yb, sigma_b,
                naive_mean=naive_mean, naive_sigma=naive_sigma,
                fgs_weight=fgs_weight,
                fgs_sigma_true=fgs_sigma_true, airs_sigma_true=airs_sigma_true
            )
            loss_b.backward()
            opt.step()

            bs = xb.size(0)
            tot_loss += loss_b.item() * bs
            tot_score += score_b.item() * bs
            n += bs

        train_loss = tot_loss / max(1, n)
        train_gll  = tot_score / max(1, n)

        # Validation
        if has_val:
            sigma_model.eval()
            with torch.no_grad():
                v_tot_loss, v_tot_score, vn = 0.0, 0.0, 0
                for xv, yv in val_dl:
                    xv = xv.to(device); yv = yv.to(device)
                    mu_v = mu_model(xv)
                    sigma_v = sigma_model(xv)
                    v_loss, v_score = normalized_gll_loss_from_sigma(
                        mu_v, yv, sigma_v,
                        naive_mean=naive_mean, naive_sigma=naive_sigma,
                        fgs_weight=fgs_weight,
                        fgs_sigma_true=fgs_sigma_true, airs_sigma_true=airs_sigma_true
                    )
                    bs = xv.size(0)
                    v_tot_loss += v_loss.item() * bs
                    v_tot_score += v_score.item() * bs
                    vn += bs
                val_loss = v_tot_loss / max(1, vn)
                val_gll  = v_tot_score / max(1, vn)
        else:
            val_loss, val_gll = None, None

        sched.step(val_loss if has_val else train_loss)

        # Save best (min loss == max GLL)
        if has_val and val_loss < best_val:
            best_val = val_loss
            torch.save({"sigma_model": sigma_model.state_dict()}, best_path)

        if (ep % 10 == 0) or (ep == 1) or (ep == num_epochs):
            if has_val:
                print(f"Epoch {ep:4d} | train_loss=-GLL {train_loss:.6e} | train_GLL {train_gll:.4f} | val_loss=-GLL {val_loss:.6e} | val_GLL {val_gll:.4f}")
            else:
                print(f"Epoch {ep:4d} | train_loss=-GLL {train_loss:.6e} | train_GLL {train_gll:.4f}")

    # Load best
    if has_val and os.path.exists(best_path):
        sigma_state = torch.load(best_path, map_location="cpu")["sigma_model"]
        sigma_model.load_state_dict(sigma_state)

    return sigma_model


In [24]:
print("Fine-tuning MLP to learn sigma-values...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mu_model, scaler = load_model('./checkpoints_gll_mean/gll_model_4.722469100215676e-07.pth')
mu_model.to(device).eval()

sigma_model = train_sigma_head(
    mu_model=mu_model,
    scaler=scaler,
    train_X=train_transit_features.astype(np.float32),
    train_Y=train_labels.astype(np.float32),
    val_X=val_transit_features.astype(np.float32),
    val_Y=val_labels.astype(np.float32),
    num_epochs=2000,
    batch_size=1024,
    lr=5e-4,
    lr_patience=200,
    ckpt_dir="./checkpoints_gll_mean"
)


Fine-tuning MLP to learn sigma-values...
Model loaded: MSE 4.722469100215676e-07, GLL 0.3515
Epoch    1 | train_loss=-GLL 4.974618e-01 | train_GLL -0.4975 | val_loss=-GLL 4.909992e-01 | val_GLL -0.4910


C:\Users\fazul\AppData\Local\Temp\ipykernel_7804\3406241572.py:255: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location='cpu'

Epoch   10 | train_loss=-GLL 4.953489e-01 | train_GLL -0.4953 | val_loss=-GLL 4.888572e-01 | val_GLL -0.4889
Epoch   20 | train_loss=-GLL 4.924448e-01 | train_GLL -0.4924 | val_loss=-GLL 4.858101e-01 | val_GLL -0.4858
Epoch   30 | train_loss=-GLL 4.881688e-01 | train_GLL -0.4882 | val_loss=-GLL 4.813209e-01 | val_GLL -0.4813
Epoch   40 | train_loss=-GLL 4.818948e-01 | train_GLL -0.4819 | val_loss=-GLL 4.747013e-01 | val_GLL -0.4747
Epoch   50 | train_loss=-GLL 4.728028e-01 | train_GLL -0.4728 | val_loss=-GLL 4.652449e-01 | val_GLL -0.4652
Epoch   60 | train_loss=-GLL 4.605057e-01 | train_GLL -0.4605 | val_loss=-GLL 4.526900e-01 | val_GLL -0.4527
Epoch   70 | train_loss=-GLL 4.460039e-01 | train_GLL -0.4460 | val_loss=-GLL 4.386083e-01 | val_GLL -0.4386
Epoch   80 | train_loss=-GLL 4.303788e-01 | train_GLL -0.4304 | val_loss=-GLL 4.224403e-01 | val_GLL -0.4224
Epoch   90 | train_loss=-GLL 4.130722e-01 | train_GLL -0.4131 | val_loss=-GLL 4.043089e-01 | val_GLL -0.4043
Epoch  100 | train_

C:\Users\fazul\AppData\Local\Temp\ipykernel_7804\1992073019.py:216: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sigma_state = torch.load(best_path, map_location="cpu")["si

In [ ]:
X_test = train_transit_features

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mu_model, scaler = load_model('./checkpoints_gll_mean/gll_model_7.087670610417263e-07.pth')
mu_model.to(device).eval()

sigma_head = FrozenTrunkSigmaHead(mu_model).to(device).eval()
sigma_state = torch.load('./checkpoints_gll_mean/sigma_head_best.pt', map_location='cpu')
state_dict = sigma_state if isinstance(sigma_state, dict) and 'sigma_model' not in sigma_state else sigma_state['sigma_model']
sigma_head.load_state_dict(state_dict)

X_test_scaled = scaler.transform(X_test).astype(np.float32)
X_test_tensor = torch.from_numpy(X_test_scaled).to(device)


with torch.no_grad():
    mu_pred = mu_model(X_test_tensor)           # (N, 283)
    sigma_pred = sigma_head(X_test_tensor)      # (N, 283)

mu_pred_np = mu_pred.cpu().numpy()
sigma_pred_np = sigma_pred.cpu().numpy()


sample_sub = pd.read_csv('./data/sample_submission.csv', index_col='planet_id')
sub = pd.DataFrame(index=pd.Series(train_indices, name='planet_id'), columns=sample_sub.columns, dtype=np.float32)
sub.iloc[:, :283]  = mu_pred_np
sub.iloc[:, 283:]  = np.clip(sigma_pred_np, 1e-15, None)
print(sub)

sub.to_csv('submission.csv')
print("Wrote /kaggle/working/submission.csv")

C:\Users\fazul\AppData\Local\Temp\ipykernel_7804\3406241572.py:255: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location='cpu'

RuntimeError: Error(s) in loading state_dict for MLPModel:
	Missing key(s) in state_dict: "net.3.weight", "net.3.bias". 
	Unexpected key(s) in state_dict: "net.5.weight", "net.5.bias", "net.2.weight", "net.2.bias". 
	size mismatch for net.0.weight: copying a param with shape torch.Size([128, 3104]) from checkpoint, the shape in current model is torch.Size([256, 3104]).
	size mismatch for net.0.bias: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([256]).